Set accelerator to T4 x2 in the UI before Save & Run All (Commit).

In [ ]:
import subprocess

print("=== /kaggle/input listing (maxdepth 4) ===")
print(subprocess.run(["find", "/kaggle/input", "-maxdepth", "4"], capture_output=True, text=True).stdout)

In [ ]:
import os
import shutil
import stat
import subprocess
from pathlib import Path

# Search for directory containing train/train_model_a.py under /kaggle/input
matches = list(Path("/kaggle/input").rglob("train/train_model_a.py"))
if len(matches) != 1:
    find_out = subprocess.run(["find", "/kaggle/input", "-maxdepth", "4"], capture_output=True, text=True).stdout
    err_msg = f"""FATAL: Expected exactly 1 match for 'train/train_model_a.py' under /kaggle/input, found {len(matches)}: {matches}.
Full tree under /kaggle/input:
{find_out}"""
    raise RuntimeError(err_msg)

code_src = matches[0].parent.parent.resolve()
print(f"Resolved code directory: {code_src}")
os.environ["SIH_CODE_DIR"] = str(code_src)

dst = Path("/kaggle/working")
print(f"Copying code from {code_src} to {dst} (including hidden files)...")

# Copy contents with copy_function=shutil.copy (not copy2) so source read-only modes are not preserved
shutil.copytree(str(code_src), str(dst), dirs_exist_ok=True, copy_function=shutil.copy)

# Explicitly ensure user write permissions on all copied files and subdirectories
for root, dirs, files in os.walk("/kaggle/working"):
    for d in dirs:
        dp = Path(root) / d
        dp.chmod(dp.stat().st_mode | stat.S_IWUSR | stat.S_IRUSR | stat.S_IXUSR)
    for f in files:
        fp = Path(root) / f
        fp.chmod(fp.stat().st_mode | stat.S_IWUSR | stat.S_IRUSR)

# Assert required files exist, failing loudly with path in message
required_code_files = [
    Path("/kaggle/working/train/train_model_a.py"),
    Path("/kaggle/working/train/dataset.py"),
    Path("/kaggle/working/configs/paths.py"),
]
for req in required_code_files:
    assert req.exists(), f"FATAL: Required code file does not exist on disk: '{req}'"

print("Code copy verified successfully. All required files present in /kaggle/working.")

# Positive write check: create and delete a temp file inside /kaggle/working/train/
probe_path = Path("/kaggle/working/train/.write_probe")
try:
    probe_path.write_text("write_probe_test", encoding="utf-8")
    assert probe_path.exists(), f"FATAL: Probe file '{probe_path}' was not written."
    probe_path.unlink()
    assert not probe_path.exists(), f"FATAL: Probe file '{probe_path}' could not be deleted."
    print("Positive write check passed: /kaggle/working/train/ is confirmed writable.")
except Exception as e:
    print(f"FATAL: Directory write permission check failed in /kaggle/working/train/: {e}")
    raise

In [ ]:
import sys
import os
from pathlib import Path

# Ensure /kaggle/working is the active working directory and on sys.path
os.chdir("/kaggle/working")
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

try:
    import configs.paths as cp
    print(f"Resolved ROOT:    {cp.ROOT}")
    print(f"Resolved SPLITS:  {cp.SPLITS}")
    print(f"Resolved CKPT:    {cp.CKPT}")
    print(f"Resolved REPORTS: {cp.REPORTS}")

    resolved_root = Path(cp.ROOT).resolve()
    expected_root = Path("/kaggle/working").resolve()
    if resolved_root != expected_root and expected_root not in resolved_root.parents:
        raise RuntimeError(f"FATAL: ROOT '{resolved_root}' resolved outside '/kaggle/working'!")

    print("SUCCESS: configs.paths imported cleanly; ROOT is within /kaggle/working.")
except Exception as e:
    print(f"FATAL: Error importing configs.paths: {e}")
    raise

In [ ]:
import os
import subprocess
import pandas as pd
from pathlib import Path

# Search for directory containing splits_v3/train.csv under /kaggle/input
matches_v3 = list(Path("/kaggle/input").rglob("splits_v3/train.csv"))
if len(matches_v3) != 1:
    find_out = subprocess.run(["find", "/kaggle/input", "-maxdepth", "4"], capture_output=True, text=True).stdout
    err_msg = f"""FATAL: Expected exactly 1 match for 'splits_v3/train.csv' under /kaggle/input, found {len(matches_v3)}: {matches_v3}.
Full tree under /kaggle/input:
{find_out}"""
    raise RuntimeError(err_msg)

splits_dir = matches_v3[0].parent.resolve()
data_dir = splits_dir.parent.resolve()

print(f"Resolved data directory:      {data_dir}")
print(f"Resolved splits_v3 directory: {splits_dir}")

# Persist to os.environ for Cell 7 (training)
os.environ["SIH_DATA_DIR"] = str(data_dir)
os.environ["SIH_SPLITS_DIR"] = str(splits_dir)

# Assert all required split files exist (including headline honest metric test_sourceheldout.csv)
data_csvs = [
    splits_dir / "train.csv",
    splits_dir / "val.csv",
    splits_dir / "class_weights.json",
    splits_dir / "test_sourceheldout.csv",
    splits_dir / "test_indist.csv",
    splits_dir / "test_external_riceblast.csv",
]
for p in data_csvs:
    assert p.exists(), f"FATAL: Split file missing on disk: '{p}'"
print("All split files (including headline honest metric test_sourceheldout.csv) confirmed present on disk.")

# Path-resolution proof: verify prefix stripping and file existence against data_dir
train_df = pd.read_csv(splits_dir / "train.csv")
raw_val = str(train_df.iloc[0]["path"])
prefix = "data/packaged_min/"
assert raw_val.startswith(prefix), f"First row path '{raw_val}' does not start with prefix '{prefix}'"
stripped_val = raw_val[len(prefix):]
joined_result = data_dir / stripped_val

print(f"Raw CSV value:   {raw_val}")
print(f"Stripped value:  {stripped_val}")
print(f"Root dir:        {data_dir}")
print(f"Joined result:   {joined_result}")

assert joined_result.exists(), f"FATAL: Joined image file does not exist on disk: '{joined_result}'"
print("Sanity check passed: first image path resolved and verified on disk.")


In [ ]:
# Pin timm==1.0.29 explicitly (verified from pip show locally in this session: 1.0.29).
# Do not use requirements-train.txt as it specifies timm>=1.0.9 which would unpin the version.
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "timm==1.0.29", "albumentations", "opencv-python-headless"],
    check=True,
)

import timm
print(f"Resolved timm version: {timm.__version__}")
assert timm.__version__ == "1.0.29", f"Expected timm 1.0.29, got {timm.__version__}"

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = wandb_api_key
    print("W&B API key loaded successfully from Kaggle Secrets.")
except Exception as e:
    print(f"W&B secret not found or unavailable ({e}); proceeding with graceful degradation.")

In [ ]:
import os
import subprocess
import sys

data_dir = os.environ["SIH_DATA_DIR"]
splits_dir = os.environ["SIH_SPLITS_DIR"]

cmd = [
    sys.executable,
    "train/train_model_a.py",
    "--data_dir", data_dir,
    "--path_prefix_strip", "data/packaged_min/",
    "--train_csv", f"{splits_dir}/train.csv",
    "--val_csv", f"{splits_dir}/val.csv",
    "--class_weights_path", f"{splits_dir}/class_weights.json",
    "--splits_dir", splits_dir,
    "--epochs", "50",
    "--batch_size", "64",
    "--device", "auto",
]

cmd_display = f"""python train/train_model_a.py   --data_dir {data_dir}   --path_prefix_strip "data/packaged_min/"   --train_csv {splits_dir}/train.csv   --val_csv {splits_dir}/val.csv   --class_weights_path {splits_dir}/class_weights.json   --splits_dir {splits_dir}   --epochs 50   --batch_size 64   --device auto"""

print("=" * 80)
print("Executing training command:")
print(cmd_display)
print("=" * 80)

subprocess.run(cmd, cwd="/kaggle/working", check=True)


In [ ]:
import json
from pathlib import Path

reports_dir = Path("/kaggle/working/artifacts/reports")
v3_report_path = reports_dir / "stage1_test_metrics.json"

assert v3_report_path.exists(), f"FATAL: Metrics report not found at {v3_report_path}"

with open(v3_report_path, "r") as f:
    metrics = json.load(f)

print("\n" + "=" * 95)
print("FINAL EVALUATION REPORT: SPLITS V3 (NEW SOURCES + ANTI-SHORTCUT AUGMENTATION)")
print("=" * 95)
header = f"{"Split / Metric":<30} | {"Split Source":<15} | {"Rows":<6} | {"Top-1 (%)":<10} | {"Macro-F1 (%)":<12} | {"Micro-F1 (%)":<12}"
print(header)
print("-" * 95)

target_splits = [
    ("val", "Validation", "splits_v3"),
    ("test_indist", "In-Distribution Test", "splits_v3"),
    ("test_sourceheldout", "Source-Heldout (Honest)", "splits_v3"),
    ("test_external_riceblast", "External Rice Blast", "splits_v3"),
]

summary_rows = []
for key, label, source_name in target_splits:
    assert key in metrics, f"FATAL: Headline metric '{key}' missing from stage1_test_metrics.json!"
    m = metrics[key]
    r = {
        "split_key": key,
        "description": label,
        "source": source_name,
        "rows": m.get("rows_evaluated", m.get("row_count", 0)),
        "top1": m["top1_accuracy"] * 100,
        "macro_f1": m["macro_f1"] * 100,
        "micro_f1": m["micro_f1"] * 100,
    }
    summary_rows.append(r)
    print(f"{r['description']:<30} | {r['source']:<15} | {r['rows']:<6} | {r['top1']:<10.2f} | {r['macro_f1']:<12.2f} | {r['micro_f1']:<12.2f}")

print("=" * 95)

final_summary_path = reports_dir / "stage1_v3_summary_table.json"
with open(final_summary_path, "w") as f:
    json.dump(summary_rows, f, indent=2)
print(f"Summary table persisted to: {final_summary_path}")
